Bloco 0
# Instalação no Windows (WSL2 + Ubuntu + Docker Desktop) — simples e direto

**Conceitos rápidos**
- **Virtualização**: recurso do processador/placa-mãe que permite rodar “máquinas” isoladas. O Docker Desktop e o WSL2 dependem disso.
- **WSL2 (Windows Subsystem for Linux 2)**: camada do Windows que executa um kernel Linux leve. O Docker Desktop usa o WSL2 “por baixo”.
- **Distribuição Linux (distro)**: um sistema Linux pronto para uso dentro do WSL2 (ex.: *Ubuntu*).

---

**1) Validar virtualização (método confiável no Windows)**
- Comando (CMD ou PowerShell):  
```bash
wmic computersystem get HypervisorPresent
```
- **Saída esperada**: `TRUE` indica que o hypervisor está ativo.  
  Observação: se aparecer `FALSE` mas **WSL2/Docker já funcionam**, ignore este teste (algumas leituras WMI são imprecisas).

**Se realmente estiver desativada**: ative **Virtualization/VT-x/AMD-V** no **BIOS/UEFI** da máquina (acesso na inicialização do PC). Não há comando seguro para ativar via terminal.

---

**2) Checar/instalar WSL2**
- Estado do WSL e versão padrão:  
```bash
wsl --status
```
- Listar distros e versões:  
```bash
wsl -l -v
```
- Se o WSL **não** estiver instalado:  
```bash
wsl --install
wsl --set-default-version 2
```  
(Reinicie quando solicitado.)

---

**3) Instalar/abrir Ubuntu**
- Instalar pelo terminal (alternativa à Microsoft Store):  
```bash
wsl --install -d Ubuntu
```
- Abrir a distro Ubuntu:  
```bash
wsl -d Ubuntu
```  
**Saída esperada**: prompt Linux (`usuario@PC:~$`). Na primeira execução, o Ubuntu pedirá para criar usuário/senha.

---

**4) Instalar o Docker Desktop (Windows)**
- Baixe e instale pelo **site oficial**. Durante a instalação, mantenha **WSL2** habilitado.  
- Abra o **Docker Desktop** e aguarde o status **Running**.

---

**5) Validar o Docker**
- Versão:  
```bash
docker --version
```
- Informações do engine:  
```bash
docker info
```  
**Saída esperada**: versão do Docker e detalhes do engine sem erros. Se falhar, abra o **Docker Desktop** e aguarde inicializar, depois rode os comandos novamente.


In [6]:
# Bloco 0 — Comandos consolidados (CMD/PowerShell)
wmic computersystem get HypervisorPresent

wsl --status
wsl -l -v

:: Instalar WSL (se necessário) e definir WSL2 como padrão
wsl --install
wsl --set-default-version 2

:: Instalar e/ou abrir o Ubuntu
wsl --install -d Ubuntu
wsl -d Ubuntu

:: Validar Docker (após instalar o Docker Desktop pelo instalador oficial)
docker --version
docker info


SyntaxError: invalid syntax (3030216484.py, line 2)

Bloco 1
# Docker: imagem, imagem base, container, build, run e porta

**Docker:** plataforma para empacotar sua aplicação com tudo o que ela precisa e executar igual em qualquer máquina.

**Imagem (image):** “pacote congelado” do app (código + bibliotecas + sistema mínimo). É criada a partir de um **Dockerfile** (a “receita”).
- Você constrói uma imagem uma vez e pode rodar em qualquer máquina com Docker.

**Imagem base (base image):** ponto de partida do Dockerfile. Ex.: `python:3.12-slim` (vem com Python + Linux mínimo).
- Por que usar `python:3.12-slim`? É leve, oficial e confiável.
- Quando criar uma base própria? Quando vários projetos compartilham **exatamente** os mesmos requisitos e/ou você quer padronizar em uma equipe.

**Container:** execução de uma imagem (um processo isolado). Você pode iniciar, parar e remover sem “sujar” sua máquina.

**Build:** transformar o Dockerfile em **imagem**.

**Run:** rodar a **imagem** como um **container**.

**Porta (port):** onde a API “escuta”.
- Dentro do container nossa API usa a porta **8010** (porta interna).
- Para acessar do seu notebook/navegador, criamos uma **ponte**: `porta_local:porta_container`.
- Com **`127.0.0.1:8010:8010`**.

**Comandos básicos de inspeção:**
- Versão do Docker:
```bash
docker --version
```

- Listar imagens salvas localmente:
```bash
docker images
```

- Listar containers (rodando) e todos (rodando/parados):
```bash
docker ps
docker ps -a
```

**Exemplos mínimos de build e run (usaremos este padrão depois):**
- Construir a imagem a partir do Dockerfile na pasta atual (nomearemos de `api_idade`):
```bash
docker build -t api_idade .
```

- Rodar a imagem, criando o container com ponte de porta local (somente sua máquina acessa):
```bash
docker run --rm --name api_idade_ct -p 127.0.0.1:8010:8010 api_idade
```


In [ ]:
# Bloco 1 — comandos consolidados
docker --version
docker images
docker ps
docker ps -a
#docker build -t api_idade .
#docker run --rm --name api_idade_ct -p 127.0.0.1:8010:8010 api_idade


Bloco 2
# Por que Docker para APIs de Machine Learning (nesta aula)

**Evitar conflitos de ambiente:** cada aluno tem versões diferentes de Python e bibliotecas; a imagem padroniza tudo.

**Reprodutibilidade:** a mesma imagem roda igual no seu notebook, no laboratório e em outra máquina.

**Empacotamento simples:** não exigimos que a pessoa “prepare” o sistema; o container já vem com o que a API precisa.

**O que vamos empacotar aqui:** A **API de previsão de idade** + o **artefato `.pkl`** do pipeline **dentro** da imagem (escolha didática para facilitar a execução local; em produção, geralmente o artefato fica fora da imagem e é baixado na inicialização).


Bloco 4
# Revisão da estrutura da API a empacotar

Vamos empacotar a **pasta da API** (ex.: `api_idade/`). Ela foi gerada pela função que cria a API pronta e contém:

- **`main.py`**  
  Ponto de entrada da FastAPI. Define `/health` e `/predict` e chama o carregamento do pipeline na inicialização.
- **`services/service.py`**  
  Carrega o **pipeline** `.pkl` (pré-processamento + modelo) e expõe a função de previsão que faz: **JSON → DataFrame → `pipeline.predict` → número**.
- **`artefatos/pipeline_referencia.pkl`**  
  **Artefato** do modelo. Colocamos **dentro** da API **apenas por didática**; em produção costuma ficar **fora** e ser baixado na subida (registry/storage).
- **`requirements.txt`**  
  Dependências **mínimas** já “limpas” (essenciais para rodar API + pipeline). As versões vêm do seu ambiente.
- **`preprocessing/`** e **`utils/`**  
  Pastas auxiliares mantidas por compatibilidade; nada a alterar para o build.
- **`models/`**  
  Pasta de compatibilidade; pode estar vazia ou com utilitários.
- **`__init__.py`** (na raiz e subpastas)  
  Torna cada diretório um **pacote** Python para os imports funcionarem.

**Porta (importante para o laboratório)**  
- Dentro do container, a API **escuta na 8010**.  
- **Fora** do container (seu Windows), você escolhe a porta **local** no `docker run` ao mapear **porta_local:porta_container**.  
  - Se o laboratório **só libera 8010**, use `127.0.0.1:8010:8010`.  
  - Se puder outra, por exemplo `127.0.0.1:8011:8010` (acessa em `http://127.0.0.1:8011`).


Bloco 5
# Dockerfile: o que é, de onde vem cada peça e como ficará

**O que é um Dockerfile?**  
É um **arquivo de receita** que ensina o Docker a montar uma **imagem** (um pacote “congelado” do seu app) em **camadas**.  
- Cada instrução (linha) vira **uma camada**.  
- Se você alterar **só o código** e não mexer nas dependências, o Docker **reaproveita as camadas antigas** e o rebuild fica bem mais rápido.

---

## 1) Imagem base — de onde vem e qual Linux é usado
- Linha no Dockerfile:
```bash
FROM python:3.12-slim
```
- **De onde vem?** Do **Docker Hub** (repositório público de imagens). **`python:3.12-slim`** é uma imagem **oficial** do time do Python.
- **Qual Linux é esse?** É um **Linux mínimo** baseado em **Debian/Bookworm (slim)**, **independente** do Ubuntu/WSL instalamos no Windows.  
  - O **WSL2** existe só para **permitir** rodar containers Linux no Windows.  
  - **Dentro do container**, quem manda é a **imagem base** que foi escolhida (aqui, Debian slim com Python 3.12).

---

## 2) Variáveis de ambiente — o que são e por que usar aqui
- Linhas no Dockerfile:
```bash
ENV PYTHONDONTWRITEBYTECODE=1 \
    PYTHONUNBUFFERED=1 \
    TF_CPP_MIN_LOG_LEVEL=3
```
- **O que são?** “Chaves” de configuração disponíveis para os processos dentro do container.
- **Aqui, para quê?**  
  - `PYTHONDONTWRITEBYTECODE=1` → o Python **não cria** arquivos `.pyc` (menos lixo na imagem).  
  - `PYTHONUNBUFFERED=1` → **logs saem na hora** (sem ficar em buffer).  
  - `TF_CPP_MIN_LOG_LEVEL=3` → o TensorFlow **silencia avisos informativos** no console (não muda os resultados do modelo, menos lixo nos logs).

---

## 3) Diretório de trabalho (WORKDIR) — “para onde” o Docker muda antes de executar comandos
- Linha no Dockerfile:
```bash
WORKDIR /app
```
- **O que faz?** Cria (se não existir) e **entra** na pasta **`/app` dentro do container**.  
- **Atenção:** `**/app**` **não** é uma pasta que exista no sistema de arquivos do computador ou da API. É **dentro da imagem**.  
  - No host (sua máquina), a pasta do projeto se chama, **`api_idade/`**.  
  - No container, vamos trabalhar **dentro de `/app`** para ter um caminho simples, padrão e previsível.

---

## 4) Copiar só o `requirements.txt` e instalar
- Linhas no Dockerfile:
```bash
COPY requirements.txt /app/
RUN pip install --no-cache-dir -r requirements.txt
```
- **O que acontece?**  
  - `COPY requirements.txt /app/` → **Copia do seu (host)** o arquivo `requirements.txt` **para /app** **dentro** da imagem.  
  - `RUN pip install ...` → **Instala as dependências** da API **dentro** da imagem.
- **Por que nessa ordem?** Se **não mudar** o `requirements.txt`, o Docker **reaproveita o cache** dessa etapa e **não reinstala tudo** em rebuilds: só reconstrói o que vier **depois** (normalmente alterações de código).

---

## 5) Copiar o restante do projeto (código + artefatos)
- Linha no Dockerfile:
```bash
COPY . /app/
```
- **O que acontece?** Copia **todo o conteúdo** da pasta onde está o Dockerfile (host), incluindo **`api_idade/`**, **`artefatos/`**, etc., **para `/app` dentro do container**.  
- **Por que não precisa existir `/app` no host?** Porque `/app` é um **caminho interno da imagem**. No host, a pasta do projeto é, por exemplo, `D:\projeto\api_idade`. No container, trabalhamos em `/app`.

---

## 6) Documentar a porta interna
- Linha no Dockerfile:
```bash
EXPOSE 8010
```
- **Para quê?** É uma **documentação** dizendo que a **aplicação** dentro do container **escuta na porta 8010**.  
- **Como acesso pelo Windows?** Quando rodarmos o container, vamos **mapear**: **porta_local:porta_do_container** (ex.: `127.0.0.1:8010:8010`).  
  - Se na faculdade **só a 8010** é liberada, mapeamos **8010:8010**.  
  - Se tiver outra porta local liberada (ex.: 8011), mapeamos **8011:8010**.

---

## 7) Como iniciar a API quando o container sobe
- Linha no Dockerfile:
```bash
CMD ["uvicorn", "api_idade.main:app", "--host", "0.0.0.0", "--port", "8010"]
```
- **O que faz?** Quando o container inicia, executa o **Uvicorn** com a sua **aplicação FastAPI** (`api_idade.main:app`) **ouvindo na 8010** e em `0.0.0.0` (todas as interfaces **dentro do container**).  
- **Fora do container**, acessamos pela **porta local** que mapeamos (ex.: `127.0.0.1:8010:8010`).

---

## Dockerfile completo (o mesmo que o Python vai gravar)

```bash
FROM python:3.12-slim

ENV PYTHONDONTWRITEBYTECODE=1 \
    PYTHONUNBUFFERED=1 \
    TF_CPP_MIN_LOG_LEVEL=3

WORKDIR /app

COPY requirements.txt /app/
RUN pip install --no-cache-dir -r requirements.txt

COPY . /app/

EXPOSE 8010

CMD ["uvicorn", "api_idade.main:app", "--host", "0.0.0.0", "--port", "8010"]
```


In [7]:
# Bloco 5 — Criar/reescrever o arquivo Dockerfile via Python (na pasta da API)

from pathlib import Path

# Ajuste se sua pasta da API tiver outro nome
pasta_api = "api_idade"

conteudo_dockerfile = """# Imagem base oficial e leve
FROM python:3.12-slim

# Variáveis de ambiente: não gerar .pyc, logs sem buffer, silenciar avisos do TensorFlow
ENV PYTHONDONTWRITEBYTECODE=1 \\
    PYTHONUNBUFFERED=1 \\
    TF_CPP_MIN_LOG_LEVEL=3

# Diretório de trabalho dentro do container
WORKDIR /app

# Copia somente as dependências e instala (camada que muda pouco)
COPY requirements.txt /app/
RUN pip install --no-cache-dir -r requirements.txt

# Copia o restante do projeto (código + artefatos)
COPY . /app/

# Porta interna em que o Uvicorn escuta
EXPOSE 8010

# Comando para iniciar a API quando o container subir
CMD ["uvicorn", "api_idade.main:app", "--host", "0.0.0.0", "--port", "8010"]
"""

caminho_dockerfile = Path(pasta_api) / "Dockerfile"
caminho_dockerfile.write_text(conteudo_dockerfile, encoding="utf-8")
print(f"Dockerfile criado/atualizado em: {caminho_dockerfile.resolve()}")

Dockerfile criado/atualizado em: C:\Users\luisasx\Downloads\aula 4\aula 4\api_idade\Dockerfile


Bloco 7
# .dockerignore: o que não deve ir para dentro da imagem

O **.dockerignore** diz ao Docker o que **não copiar** para dentro da imagem. Isso:
- **Reduz o tamanho** da imagem (não leva arquivos inúteis).
- **Acelera o build** (menos arquivos para enviar e processar).
- **Evita “lixo”** de ambiente (venvs locais, caches, checkpoints, etc.).

Incluir entradas típicas de “tralha” e de coisas **que a API não usa em produção**:
- Ambientes virtuais e caches: `.venv/`, `venv/`, `env/`, `__pycache__/`, `*.pyc`, `*.pyo`, `*.pyd`, `.pytest_cache/`, `.mypy_cache/`, `.cache/`
- Pastas temporárias/artefatos de build: `build/`, `dist/`
- Checkpoints de notebooks: `.ipynb_checkpoints/`
- Pastas de IDEs e metadados: `.idea/`, `.vscode/`, `.git/`, `.gitignore`
- Arquivos do SO: `.DS_Store`, `Thumbs.db`
- Dados grandes que **não** são necessários para **servir** a API: `data/`, `dados/`, `datasets/`, `logs/`
- Arquivos de config sensíveis que **não** devem ir para a imagem: `.env`, `.env.*`

**Atenção:** não ignore o que a API **precisa** para rodar (ex.: `requirements.txt`, código da API, e **o artefato** `artefatos/pipeline_referencia.pkl`).


In [8]:
# Bloco 7
# Criar/reescrever um .dockerignore básico na pasta da API

from pathlib import Path

pasta_api = "api_idade"  # ajuste se sua pasta tiver outro nome

conteudo_dockerignore = """# Ambientes virtuais / caches Python
.venv/
venv/
env/
__pycache__/
*.pyc
*.pyo
*.pyd
.pytest_cache/
.mypy_cache/
.cache/

# Artefatos de build / distribuição
build/
dist/

# Notebooks e checkpoints
.ipynb_checkpoints/

# IDEs e VCS
.idea/
.vscode/
.git/
.gitignore

# Arquivos do SO
.DS_Store
Thumbs.db

# Pastas/arquivos grandes não usados pela API em produção
data/
dados/
datasets/
logs/

# Variáveis de ambiente (não embutir segredos na imagem)
.env
.env.*
"""

caminho_dockerignore = Path(pasta_api) / ".dockerignore"
caminho_dockerignore.write_text(conteudo_dockerignore.strip() + "\n", encoding="utf-8")

print(f".dockerignore criado/atualizado em: {caminho_dockerignore.resolve()}")


.dockerignore criado/atualizado em: C:\Users\luisasx\Downloads\aula 4\aula 4\api_idade\.dockerignore


Bloco 9
# Build: construir a imagem da API (onde rodar e o que esperar)

**Onde rodar os comandos de `docker`:**  
- **Windows:** pode usar **PowerShell**, **Windows Terminal**, **CMD** **ou** o **Anaconda Prompt** (tanto faz).  
  - **Não precisa** ativar nenhum ambiente virtual para o *build* — o Docker constrói tudo **dentro da imagem**, isolado do seu Python local.  
- **WSL (Ubuntu):** também funciona, mas **não é necessário**. No Windows com Docker Desktop “Running”, usar PowerShell/Anaconda Prompt já resolve.

**Em qual pasta rodar:**  
- Entre na **pasta da API** (ex.: `api_idade/`), a **mesma** onde está o arquivo **`Dockerfile`**.  
- O **`.`** no final do comando significa “use **esta pasta** como contexto do build”.

**O que acontece no primeiro vs. próximos builds:**  
- **Primeiro build:** demora mais (baixa a imagem base `python:3.12-slim` e instala as dependências do `requirements.txt`).  
- **Builds seguintes:** mais rápidos, porque o Docker **reaproveita as camadas** que **não mudaram** (cache).  
  - Se você **só** alterar o **código da API** e **não** mexer no `requirements.txt`, a etapa de instalar libs **fica em cache** e o rebuild é bem mais rápido.

**Comandos (na pasta da API):**  
```bash
# 1) (Opcional) verifique se o Dockerfile está aqui
dir          # Windows (ou 'ls' no WSL)

# 2) construir a imagem chamada 'api_idade'
docker build -t api_idade .

# 3) listar imagens locais (confirme se 'api_idade' apareceu)
docker images
```

**Dicas úteis:**  
- Se quiser forçar um build **ignorando cache** (por ex., trocou dependências e quer garantir instalação do zero):  
```bash
docker build --no-cache -t api_idade .
```
- Se aparecer erro de “não encontrou Dockerfile”, confira se você está na **pasta correta** (onde o Dockerfile foi criado).


In [ ]:
# Bloco 9 — Comandos consolidados (Terminal / PowerShell)

# entrar na pasta da API (ajuste se seu nome for outro)
cd api_idade

# construir a imagem com nome 'api_idade'
docker build -t api_idade .

# listar imagens para confirmar
docker images

Bloco 9.1
# Depois do **build**: o que foi criado (imagem) e como reaproveitar

## O que o `docker build` acabou de criar
- Ele montou uma **imagem Docker**: um “pacote congelado” e **imutável** com tudo que a API precisa para rodar.
- No nosso caso, a imagem contém:
  - Um **Linux mínimo** (o mesmo usado pela imagem base `python:3.12-slim`, que é Debian slim).
  - O **Python 3.12** já instalado.
  - Todas as **bibliotecas** do `requirements.txt`.
  - A **pasta do projeto** (`api_idade/`) e o **artefato** `artefatos/pipeline_referencia.pkl`.
  - O **comando de entrada** (Uvicorn) definido no `Dockerfile` (`CMD [...]`).

> Pense na imagem como uma **foto do seu app** em camadas. Ela não “roda” sozinha; para executar precisamos cria um **container** a partir dela (próximo bloco).

## Onde essa imagem fica
- Ela fica no **cache local do Docker** (não vira automaticamente um arquivo no seu disco).
- Consegue **listar** com:  
  ```bash
  docker images
  ```

## Dá para transformar a imagem em arquivo?
- Sim. Podemos **exportar** a imagem para um arquivo `.tar` e levar para outro computador (pen drive, rede, etc.):
  - **Salvar em .tar**:  
    ```bash
    docker save -o api_idade.tar api_idade
    ```
  - **Carregar de volta** (no mesmo PC ou em outro):  
    ```bash
    docker load -i api_idade.tar
    ```

## O que fazer com a imagem
- **Rodar** a API criando um **container**: veremos no próximo bloco (`docker run -p ... api_idade`).
- **Reutilizar** em outras máquinas (Windows, macOS, Linux) sem “dor de dependências”.

## Posso reaproveitar para **outros projetos**?
- Sim, mas com **cuidado**:
  - A nossa imagem contém **esta API específica** + seu `.pkl`. Não é uma “base genérica”.
  - Para padronizar times, o ideal é criar uma **imagem base da equipe** (ex.: só Linux + Python + NumPy/pandas/TensorFlow com versões fixas).  
    Depois, nos projetos, o `Dockerfile` começa com `FROM sua_base:versao` e adiciona **apenas o código do projeto**.
- Para **nossa aula**, vamos usar **esta imagem** só para rodar e testar a API de idade.

## Resumo prático
- O **build** criou uma **imagem** pronta para rodar **igual** em qualquer lugar.
- Você pode **rodar** (próximo bloco), **exportar** (`docker save`) ou **compartilhar** via registry.
- Para **padronizar** projetos futuros, pense em uma **imagem base da equipe** e use essa base nos `Dockerfile` dos projetos.


In [ ]:
# Bloco 9.1 — Comandos

# listar imagens locais (ver se "api_idade" existe)
docker images

# (opcional) exportar a imagem como arquivo .tar para levar a outro PC/servidor
docker save -o api_idade.tar api_idade

# (opcional) importar a imagem .tar em outra máquina (ou na mesma)
docker load -i api_idade.tar


Bloco 10
# Run: rodar o container na máquina

**Onde rodar:** pode usar **PowerShell**, **Windows Terminal**, **CMD** ou **Anaconda Prompt** no Windows (tanto faz). Não precisa do WSL para este passo.

## O que é “porta” e “ponte de porta”
- A **API dentro do container** está configurada para escutar **lá dentro** na porta **8010** (isso veio do nosso Dockerfile com `--port 8010`).
- Para acessar do **navegador** ou **notebook** no Windows, criamos uma **ponte** entre uma porta do **computador** e a porta **dentro do container**.
- Essa ponte é escrita assim: **porta_do_PC : porta_do_container**.
- Quando incluímos **`127.0.0.1:`** antes, estamos dizendo: **só esta máquina** pode acessar (não expõe na rede da faculdade).

Exemplos de mapeamento:
- `127.0.0.1:8010:8010` → sua máquina acessa **http://127.0.0.1:8010** e isso cai na **porta 8010 do container**.
- Se a sua porta **8010** local já estiver ocupada, troque a **porta do seu PC** (a da esquerda), por exemplo: `127.0.0.1:8011:8010`.  
  Dentro do container **continua 8010**. Por fora você acessa **http://127.0.0.1:8011**.

## Explicando o comando
- `docker run` → cria e inicia um **container novo** a partir da imagem.
- `--rm` → **remove** o container automaticamente quando ele for encerrado.
- `--name api_idade_ct` → dá um **nome fácil** para o container (útil para ver logs, parar etc.).
- `-p 127.0.0.1:8010:8010` → **publica a porta**:
  - **127.0.0.1** = **bind local** (só seu PC acessa).
  - **8010 (esquerda)** = porta do **seu PC** (onde você acessa).
  - **8010 (direita)** = porta **dentro do container** (onde o Uvicorn está escutando).
- `api_idade` → é o **nome da imagem** que você construiu no Bloco 9.

## Rodar o container apenas local
```bash
docker run --rm --name api_idade_ct -p 127.0.0.1:8010:8010 api_idade
```

Se a **porta 8010 do seu PC** estiver ocupada, use outra (ex.: **8011**):
```bash
docker run --rm --name api_idade_ct -p 127.0.0.1:8011:8010 api_idade
```

**Durante a execução:** os **logs** aparecem no terminal.  
**Para encerrar:** pressione **CTRL+C** (como usamos `--rm`, o container é removido ao sair).  
Depois de subir, você pode testar o endpoint **/health** no navegador: `http://127.0.0.1:8010/health` (ou `:8011` se tiver trocado).


In [ ]:
# Bloco 10 — Comandos

# rodar o container, acessível só pela sua máquina (porta local 8010 -> porta 8010 do container)
docker run --rm --name api_idade_ct -p 127.0.0.1:8010:8010 api_idade

# (alternativa) se a porta 8010 do seu PC estiver ocupada, use 8011 por fora:
# docker run --rm --name api_idade_ct -p 127.0.0.1:8011:8010 api_idade

# (dica) depois de subir, teste no navegador:
# http://127.0.0.1:8010/health
# (ou http://127.0.0.1:8011/health se mapeou 8011)


# Erro proposital e correção do Dockerfile

## O que aconteceu (e por que apareceu `ModuleNotFoundError: No module named 'api_idade'`)
Fizemos o **build** estando **dentro da pasta** `api_idade/`. No `Dockerfile` original, esta linha:
```bash
COPY . /app/
```
copiou **tudo que está na pasta atual** (do projeto) **direto para** `/app` **dentro do container**.  
Resultado: os arquivos ficaram como `/app/main.py`, `/app/services/service.py`, etc.  
Só que o comando de inicialização do Uvicorn tenta abrir **um pacote** chamado `api_idade`:
```bash
uvicorn api_idade.main:app
```
e **não existe** uma pasta `api_idade/` (com `__init__.py`) em `/app` — os arquivos foram colocados um nível acima. Por isso o Python não encontra o módulo `api_idade` e dispara o erro.

> Analogia rápida: pense no Python procurando uma **caixa** chamada `api_idade` com o arquivo `main.py` dentro. Colocamos tudo **fora** da caixa, então ele não acha a caixa.

---

## Forma correta (manter o pacote `api_idade` dentro do container)
Precisamos **preservar a pasta** `api_idade/` **dentro do container**, para que o módulo exista.  
A correção é simples: **copiar o projeto para** `/app/api_idade` (e não para `/app`). Assim, dentro do container teremos a caixa `api_idade/` com `main.py` lá dentro, e o comando continua funcionando:

```bash
uvicorn api_idade.main:app
```

### Dockerfile corrigido (bloco completo)
```bash
FROM python:3.12-slim

ENV PYTHONDONTWRITEBYTECODE=1 \
    PYTHONUNBUFFERED=1 \
    TF_CPP_MIN_LOG_LEVEL=3

WORKDIR /app

# Instala dependências primeiro (camada estável, aproveita cache)
COPY requirements.txt /app/
RUN pip install --no-cache-dir -r requirements.txt

# Agora copia o seu projeto mantendo o pacote 'api_idade' visível para o Python
COPY . /app/api_idade

# Porta interna usada pelo Uvicorn dentro do container
EXPOSE 8010

# Sobe a API apontando para o pacote 'api_idade'
CMD ["uvicorn", "api_idade.main:app", "--host", "0.0.0.0", "--port", "8010"]
```

In [9]:
from pathlib import Path

# Nome da pasta da API (ajuste se a sua API tiver outro nome de pacote)
pasta_api = "api_idade"

# Caminho do Dockerfile
caminho_dockerfile = Path(pasta_api) / "Dockerfile"

# Conteúdo corrigido do Dockerfile (camadas em ordem para aproveitar cache)
conteudo_dockerfile = """# Imagem base oficial do Python (leve)
FROM python:3.12-slim

# Variáveis de ambiente: não gerar .pyc, logs sem buffer, silenciar avisos do TensorFlow
ENV PYTHONDONTWRITEBYTECODE=1 \\
    PYTHONUNBUFFERED=1 \\
    TF_CPP_MIN_LOG_LEVEL=3

# Diretório de trabalho dentro do container
WORKDIR /app

# Copia apenas as dependências e instala (camada estável reaproveitada pelo cache)
COPY requirements.txt /app/
RUN pip install --no-cache-dir -r requirements.txt

# Copia o restante do projeto preservando o pacote 'api_idade' visível para o Python
COPY . /app/api_idade

# Porta interna onde o Uvicorn escuta
EXPOSE 8010

# Comando para iniciar a API quando o container subir
CMD ["uvicorn", "api_idade.main:app", "--host", "0.0.0.0", "--port", "8010"]
"""

# Grava o Dockerfile
caminho_dockerfile.write_text(conteudo_dockerfile, encoding="utf-8")

print(f"Dockerfile criado/atualizado em: {caminho_dockerfile.resolve()}")


Dockerfile criado/atualizado em: C:\Users\luisasx\Downloads\aula 4\aula 4\api_idade\Dockerfile


# Rebuild rápido: por que agora é bem mais rápido e o que rodar

**O que mudou?** Nós só corrigimos **a última linha do Dockerfile** (`CMD ...`).  
As **camadas** anteriores (imagem base, `ENV`, `WORKDIR`, `COPY requirements.txt`, `RUN pip install`, `COPY .`, `EXPOSE`) **não mudaram**.  
Por isso, no `docker build` o Docker **reaproveita o cache** até o ponto em que nada mudou (principalmente a etapa de **instalar dependências**).  
Resultado: o rebuild agora é **muito mais rápido** (ele pula o download da base e a reinstalação das libs).

**Onde rodar os comandos?**  
No **Windows**, tanto faz **PowerShell**, **Windows Terminal** ou **Anaconda Prompt** — o importante é o **Docker Desktop estar aberto** e você estar **dentro da pasta da API** (ex.: `api_idade/`, onde está o `Dockerfile`).

**Passo a passo (rebuild + subir o container):**
1) **Rebuild da imagem** (reaproveitando cache das dependências):  
```bash
docker build -t api_idade .
```

2) **(Opcional) Conferir a imagem** criada/atualizada:  
```bash
docker images
```

3) **Subir o container só para sua máquina** (porta local **8010** → porta **8010** do container):  
```bash
docker run --rm --name api_idade_ct -p 127.0.0.1:8010:8010 api_idade
```

> Dica: se a **porta 8010** já estiver em uso no seu Windows, troque a **porta local** (ex.: `8011:8010`):  
> ```bash  
> docker run --rm --name api_idade_ct -p 127.0.0.1:8011:8010 api_idade  
> ```

**Como parar o container:** pressione **CTRL+C** no terminal. Com `--rm`, o container é removido automaticamente ao encerrar.


In [ ]:
# Bloco B — comandos consolidados (rebuild + subir container)

# 1) Rebuild da imagem (usa cache das dependências)
docker build -t api_idade .

# 2) (Opcional) Conferir a imagem criada/atualizada
docker images

# 3) Subir o container acessível só pela sua máquina (porta local 8010 → porta 8010 do container)
docker run --rm --name api_idade_ct -p 127.0.0.1:8010:8010 api_idade

# (Opcional) Se a porta 8010 local estiver ocupada, use outra porta local (ex.: 8011)
# docker run --rm --name api_idade_ct -p 127.0.0.1:8011:8010 api_idade


SyntaxError: invalid syntax (1069503184.py, line 4)

Bloco 11
# Testes rápidos: **/health** e **/predict**

**Como checar se a API está de pé:**  
Abra o navegador e acesse **http://127.0.0.1:8010/health**.  
Se tudo estiver ok, você verá um JSON simples, `{"status":"ok"}`. Isso confirma que o container está rodando e o servidor Uvicorn iniciou.

**Como vamos fazer uma previsão:**  
1) Ler o CSV usado no treino (ou poderia ser outro arquivo com as mesmas colunas).  
2) Pegar uma linha pelo **índice**.  
3) Montar o **payload** removendo a coluna alvo (`age`) — ou seja, vão só as **mesmas chaves** usadas no treino.  
4) Enviar esse payload para `http://127.0.0.1:8010/predict`.  
5) Mostrar na tela o **valor real** (do CSV) e o **valor previsto** (sem casas decimais).

In [13]:
pip install requests

  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached charset_normalizer-3.4.3-cp312-cp312-win_amd64.whl.metadata (37 kB)
  Using cached urllib3-2.5.0-py3-none-any.whl.metadata (6.5 kB)
Using cached requests-2.32.5-py3-none-any.whl (64 kB)
Using cached charset_normalizer-3.4.3-cp312-cp312-win_amd64.whl (107 kB)
Using cached urllib3-2.5.0-py3-none-any.whl (129 kB)

   ---------------------------------------- 0/3 [urllib3]
   ---------------------------------------- 0/3 [urllib3]
   ---------------------------------------- 0/3 [urllib3]
   ---------------------------------------- 0/3 [urllib3]
   ------------- -------------------------- 1/3 [charset_normalizer]
   ------------- -------------------------- 1/3 [charset_normalizer]
   -------------------------- ------------- 2/3 [requests]
   ---------------------------------------- 3/3 [requests]

Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, which is not installed.


In [1]:
pip install numpy pandas scikit-learn scikeras tensorflow joblib scipy cloudpickle

  Using cached scikeras-0.13.0-py3-none-any.whl.metadata (3.1 kB)
Using cached scikeras-0.13.0-py3-none-any.whl (26 kB)
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import requests


def extrair_payload_y(tabela, indice, coluna_alvo="age"):
    """
    Retorna (payload_sem_alvo, y_real) para a linha indicada.
    """
    linha = tabela.iloc[indice]                          # pega a linha pelo índice
    y_real = float(linha[coluna_alvo])                   # valor real do alvo
    payload = linha.drop(labels=[coluna_alvo]).to_dict() # remove o alvo do payload
    payload = {k: (v.item() if hasattr(v, "item") else v) for k, v in payload.items()}  # tipagem nativa p/ JSON
    return payload, y_real


def prever_api(payload, y_real, base_url="http://127.0.0.1:8010"):
    """
    Envia payload à API (/predict) e mostra y_real e y_pred (sem casas decimais).
    """
    url = f"{base_url}/predict"                          # endpoint da API
    try:
        resp = requests.post(url, json=payload, timeout=30)  # faz o POST
        data = resp.json() if resp.ok else {}                # extrai JSON se OK
        y_pred = float(data.get("idade_prevista", np.nan))   # pega a predição
    except Exception:
        y_pred = np.nan                                      # evita erro na tela

    y_pred_int = int(round(y_pred)) if np.isfinite(y_pred) else None  # sem casas decimais
    print(f"y_real = {y_real} | y_pred = {y_pred_int}")                    # mostra os valores

CAMINHO_CSV = "dados/fuma_e_bebe.csv"
df = pd.read_csv(CAMINHO_CSV)

In [ ]:
payload, y_real = extrair_payload_y(df, 5)
prever_api(payload, y_real)

y_real = 35.0 | y_pred = 38


Bloco 10B
# Rodar sem “prender” o terminal, ver logs e limpar tudo

**Comando base (explicado):**
```bash
docker run --rm --name api_idade_ct -p 127.0.0.1:8010:8010 api_idade
```

- `docker run` → cria **e** inicia um **container** a partir de uma **imagem**.
- `--rm` → quando o container **parar**, ele é **apagado automaticamente** (útil em aula para não acumular “lixo”).
- `--name api_idade_ct` → dá um **nome amigável** ao container.  
  - Não há padrão obrigatório; aqui usei `api_idade_ct` só para “API de idade – conTainer”. Pode escolher outro.
- `-p 127.0.0.1:8010:8010` → cria uma **ponte de portas**:
  - **antes dos dois-pontos**: `127.0.0.1:8010` = **sua máquina** e **porta local** (só você acessa).
  - **depois**: `8010` = **porta interna** do container onde a API escuta.
  - Resultado: acessar `http://127.0.0.1:8010` do seu navegador chega na API dentro do container.
- `api_idade` → **nome da imagem** que você construiu no `docker build`.

**Problema desse comando do jeito que está:** ele roda em **primeiro plano** (“**grudado**” no terminal), mostrando os logs ali. Enquanto estiver rodando, seu terminal fica **ocupado**.  
Se você quiser continuar usando o terminal, rode em **segundo plano**:

**Subir em segundo plano (detached):**
```bash
docker run -d --rm --name api_idade_ct -p 127.0.0.1:8010:8010 api_idade
```
- `-d` (**detached**) → sobe o container e **libera o terminal** imediatamente.

**Ver os logs quando quiser:**
- Últimas linhas (estático):  
  ```bash
  docker logs api_idade_ct
  ```
- Ao vivo (“seguir”):  
  ```bash
  docker logs -f api_idade_ct
  ```
  - Use **Ctrl+C** para **sair da visualização**; o container **continua rodando**.

**Parar e remover o container:**
- Parar:  
  ```bash
  docker stop api_idade_ct
  ```
- Remover manualmente (só se **não** usou `--rm`):  
  ```bash
  docker rm api_idade_ct
  ```

**Outros comandos úteis (claros e diretos):**
- Ver containers **rodando** agora:  
  ```bash
  docker ps
  ```
- Ver **todos** os containers (inclusive parados):  
  ```bash
  docker ps -a
  ```
- Ver as **imagens** que existem na sua máquina:  
  ```bash
  docker images
  ```
- Apagar uma **imagem** (se nenhum container estiver usando):  
  ```bash
  docker rmi api_idade
  ```
- Forçar remoção (cuidado, derruba o que estiver usando):  
  ```bash
  docker rm -f api_idade_ct
  ```
  ```bash
  docker rmi -f api_idade
  ```

**Dica de porta ocupada:** se `127.0.0.1:8010` já estiver em uso, troque **só a porta local**, por exemplo:
```bash
docker run -d --rm --name api_idade_ct -p 127.0.0.1:8011:8010 api_idade
```
Acesse então `http://127.0.0.1:8011/health`.


In [ ]:
# Bloco 10B — sequência de comandos (do zero até limpar tudo)

# --- MODO 1: rodar EM PRIMEIRO PLANO (terminal “preso” nos logs) ---
# Subir o container (sai com Ctrl+C; com --rm ele apaga ao parar)
docker run --rm --name api_idade_ct -p 127.0.0.1:8010:8010 api_idade

# (Depois que sair com Ctrl+C, o container foi parado e removido por causa do --rm)
docker ps            # não deve aparecer o container
docker ps -a         # também não deve aparecer (foi removido)

# --- MODO 2: rodar EM SEGUNDO PLANO (detached) ---
docker run -d --rm --name api_idade_ct -p 127.0.0.1:8010:8010 api_idade
docker ps            # deve listar api_idade_ct como "Up"
docker ps -a         # idem, mostra os rodando e parados

# Ver logs uma vez (estático) e ao vivo (sair com Ctrl+C)
docker logs api_idade_ct
docker logs -f api_idade_ct

# Parar o container (como usamos --rm, ao parar ele já será removido)
docker stop api_idade_ct
docker ps            # vazio
docker ps -a         # não deve listar api_idade_ct (foi removido)

# --- MODO 3: rodar sem --rm (para ver a diferença) ---
docker run -d --name api_idade_ct -p 127.0.0.1:8010:8010 api_idade
docker ps            # container rodando
docker stop api_idade_ct
docker ps -a         # agora aparece PARADO (Exited), porque NÃO usamos --rm
docker rm api_idade_ct
docker ps -a         # some da lista

# --- Ver e limpar imagens ---
docker images        # ver a imagem "api_idade"
docker rmi api_idade # remove a imagem (se não houver container usando)
# Se der erro de “in use”, pare/remoção o container ou force:
# docker rm -f api_idade_ct
# docker rmi -f api_idade


Bloco 13
# Ciclo de desenvolvimento: quando rebuildar e como

**Casos comuns e o que fazer:**

**A) Mudou só o código da API (`.py`)**  
- O cache de dependências é reaproveitado. Rebuild **rápido**.  
- Passos:
```bash
docker stop api_idade_ct
docker rm api_idade_ct
docker build -t api_idade .
docker run --rm --name api_idade_ct -p 127.0.0.1:8010:8010 api_idade
```

**B) Mudou o `requirements.txt` (adicionou/removeu libs)**  
- A camada de dependências precisa ser refeita. Use `--no-cache` para garantir reinstalação limpa.  
- Passos:
```bash
docker stop api_idade_ct
docker rm api_idade_ct
docker build --no-cache -t api_idade .
docker run --rm --name api_idade_ct -p 127.0.0.1:8010:8010 api_idade
```

**C) Trocou apenas o artefato `.pkl`**  
- O COPY do projeto muda; dependências permanecem em cache. Rebuild **normal** (rápido).  
- Passos:
```bash
docker stop api_idade_ct
docker rm api_idade_ct
docker build -t api_idade .
docker run --rm --name api_idade_ct -p 127.0.0.1:8010:8010 api_idade
```